#Cell 1: Imports and config

In [ ]:
import json
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression

SEED = 42
SIM_THRESHOLD = 0.88
N_SPLITS = 5

TRAIN_PATH = "train.csv"
VAL_PATH = "val.csv"
TEST_PATH = "test.csv"

TEXT_COL = "description"
LABEL_COL = "label"

OUTPUT_DIR = Path("tfidf_cluster_aware_outputs")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

#Cell 2: Load and merge data

In [ ]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).strip().lower()

    # normalize spaces
    text = re.sub(r"\s+", " ", text)

    # remove repeated punctuation spacing
    text = re.sub(r"[^\w\sàáảãạăắằẳẵặâấầẩẫậèéẻẽẹêếềểễệìíỉĩị"
                  r"òóỏõọôốồổỗộơớờởỡợùúủũụưứừửữựỳýỷỹỵđ]", " ", text)

    # normalize spaces again
    text = re.sub(r"\s+", " ", text).strip()
    return text

def load_split(path, split_name):
    df = pd.read_csv(path)
    expected_cols = {TEXT_COL, LABEL_COL}
    missing = expected_cols - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {missing}")

    df = df[[TEXT_COL, LABEL_COL]].copy()
    df[TEXT_COL] = df[TEXT_COL].apply(normalize_text)
    df[LABEL_COL] = df[LABEL_COL].astype(str)
    df["source_split"] = split_name
    return df

train_df = load_split(TRAIN_PATH, "train")
val_df = load_split(VAL_PATH, "val")
test_df = load_split(TEST_PATH, "test")

df = pd.concat([train_df, val_df, test_df], ignore_index=True).reset_index(drop=True)

# keep all rows, but create exact-normalized key for inspection
df["exact_key"] = df[TEXT_COL]
df["sample_id"] = np.arange(len(df))

print("Merged dataset shape:", df.shape)
print("\nLabel distribution:")
print(df[LABEL_COL].value_counts())
print("\nSource split distribution:")
print(df["source_split"].value_counts())

exact_dup_counts = df["exact_key"].value_counts()
print("\nExact duplicate text groups:", int((exact_dup_counts > 1).sum()))
print("Exact duplicate rows:", int((exact_dup_counts[exact_dup_counts > 1].sum())))
df.head()

Merged dataset shape: (6606, 5)

Label distribution:
label
3    2415
2    1843
4    1350
1     998
Name: count, dtype: int64

Source split distribution:
source_split
train    4624
val       991
test      991
Name: count, dtype: int64

Exact duplicate text groups: 1
Exact duplicate rows: 2


,description,label,source_split,exact_key,sample_id
0,một chút mỏi mắt muốn nhắm mắt sau 2 3 tiếng,2,train,một chút mỏi mắt muốn nhắm mắt sau 2 3 tiếng,0
1,cảm thấy mệt mắt hay nặng mí phải nghỉ giữa chừng,3,train,cảm thấy mệt mắt hay nặng mí phải nghỉ giữa chừng,1
2,thường xuyên mệt mắt nhức nhức làm kém hiệu quả,3,train,thường xuyên mệt mắt nhức nhức làm kém hiệu quả,2
3,cộm mắt rất nhiều mắt sưng to nghỉ rất lâu,4,train,cộm mắt rất nhiều mắt sưng to nghỉ rất lâu,3
4,thỉnh thoảng nặng mí sưng nhẹ,2,train,thỉnh thoảng nặng mí sưng nhẹ,4


#Cell 3: Build similarity clusters

In [ ]:
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import coo_matrix

# stricter thresholds than before
WORD_THRESHOLD = 0.82
CHAR_THRESHOLD = 0.88
TOP_K = 30

texts = df[TEXT_COL].tolist()
n_samples = len(texts)

# 1) word-level TF-IDF
word_vectorizer = TfidfVectorizer(
    lowercase=False,
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)

X_word = word_vectorizer.fit_transform(texts)
print("Word TF-IDF shape:", X_word.shape)

# 2) char-level TF-IDF (better for near-duplicate / paraphrase-ish variations)
char_vectorizer = TfidfVectorizer(
    lowercase=False,
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.98,
    sublinear_tf=True,
)

X_char = char_vectorizer.fit_transform(texts)
print("Char TF-IDF shape:", X_char.shape)

# 3) exact duplicates must always be grouped
exact_groups = {}
for idx, key in enumerate(df["exact_key"].tolist()):
    exact_groups.setdefault(key, []).append(idx)

rows = []
cols = []
data = []

# self-loops
for i in range(n_samples):
    rows.append(i)
    cols.append(i)
    data.append(1)

# add exact-duplicate edges
for _, idxs in exact_groups.items():
    if len(idxs) > 1:
        for i in idxs:
            for j in idxs:
                rows.append(i)
                cols.append(j)
                data.append(1)

# 4) kNN on word space
nn_word = NearestNeighbors(
    n_neighbors=min(TOP_K, n_samples),
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)
nn_word.fit(X_word)
dist_word, ind_word = nn_word.kneighbors(X_word)

# 5) kNN on char space
nn_char = NearestNeighbors(
    n_neighbors=min(TOP_K, n_samples),
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)
nn_char.fit(X_char)
dist_char, ind_char = nn_char.kneighbors(X_char)

# 6) build stricter graph:
# connect if word similarity high enough OR char similarity high enough
for i in range(n_samples):
    # word edges
    for d, j in zip(dist_word[i], ind_word[i]):
        sim = 1.0 - float(d)
        if i != j and sim >= WORD_THRESHOLD:
            rows.append(i)
            cols.append(j)
            data.append(1)

    # char edges
    for d, j in zip(dist_char[i], ind_char[i]):
        sim = 1.0 - float(d)
        if i != j and sim >= CHAR_THRESHOLD:
            rows.append(i)
            cols.append(j)
            data.append(1)

adjacency = coo_matrix((data, (rows, cols)), shape=(n_samples, n_samples)).tocsr()
adjacency = adjacency.maximum(adjacency.T)

n_components, cluster_ids = connected_components(
    csgraph=adjacency,
    directed=False,
    return_labels=True
)

df["cluster_id"] = cluster_ids

cluster_sizes = pd.Series(cluster_ids).value_counts().sort_values(ascending=False)

print("\n===== Strict Cluster Summary =====")
print("Number of clusters:", n_components)
print("Largest 20 cluster sizes:")
print(cluster_sizes.head(20))

print("\nCluster size summary:")
print(cluster_sizes.describe())

singleton_ratio = (cluster_sizes == 1).mean()
print(f"\nSingleton cluster ratio: {singleton_ratio:.4f}")

# quick audit: show a few big clusters
print("\n===== Sample large clusters =====")
top_clusters = cluster_sizes.head(5).index.tolist()
for cid in top_clusters:
    print(f"\nCluster {cid} | size = {cluster_sizes[cid]}")
    examples = df.loc[df["cluster_id"] == cid, TEXT_COL].head(5).tolist()
    for ex in examples:
        print(" -", ex)

Word TF-IDF shape: (6606, 1603)
Char TF-IDF shape: (6606, 1499)

===== Strict Cluster Summary =====
Number of clusters: 3466
Largest 20 cluster sizes:
51     82
184    79
78     67
400    46
101    41
352    39
85     39
43     37
83     36
99     36
260    33
237    31
307    29
103    28
4      28
115    27
129    26
75     25
88     24
294    23
Name: count, dtype: int64

Cluster size summary:
count    3466.000000
mean        1.905943
std         3.602842
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        82.000000
Name: count, dtype: float64

Singleton cluster ratio: 0.7268

===== Sample large clusters =====

Cluster 51 | size = 82
 - cảm thấy nặng nề hay nặng mí phải giảm thời gian màn hình
 - rất mỏi mắt hay khô mắt nhiều phải giảm thời gian màn hình
 - thường xuyên nặng nề khô mắt nhiều làm phải giảm thời gian màn hình
 - cảm thấy nhức mắt thường xuyên nhức nhức phải giảm thời gian màn hình
 - mỏi mắt đáng kể cảm giác cộm nên phải giảm

#Cell 4: Cluster-aware 5-fold CV

In [ ]:
X_text = df[TEXT_COL].values
y = df[LABEL_COL].values
groups = df["cluster_id"].values

cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

fold_metrics = []
oof_rows = []

for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X_text, y, groups=groups), start=1):
    X_train_text = X_text[train_idx]
    X_test_text = X_text[test_idx]
    y_train = y[train_idx]
    y_test = y[test_idx]

    fold_vectorizer = TfidfVectorizer(
        lowercase=False,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
    )

    X_train = fold_vectorizer.fit_transform(X_train_text)
    X_test = fold_vectorizer.transform(X_test_text)

    clf = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=SEED,
        solver="liblinear",
    )

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_test, y_pred, average="weighted", zero_division=0
    )

    # leakage audit inside each fold:
    # measure max train-test cosine similarity on word tf-idf of this fold
    fold_audit_vectorizer = TfidfVectorizer(
        lowercase=False,
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
    )
    A_train = fold_audit_vectorizer.fit_transform(X_train_text)
    A_test = fold_audit_vectorizer.transform(X_test_text)

    sim_matrix = cosine_similarity(A_test, A_train, dense_output=False)
    max_sim_per_test = np.asarray(sim_matrix.max(axis=1).toarray()).ravel()

    leakage_090 = float((max_sim_per_test >= 0.90).mean())
    leakage_095 = float((max_sim_per_test >= 0.95).mean())
    mean_max_sim = float(max_sim_per_test.mean())

    fold_metrics.append({
        "fold": fold_idx,
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "clusters_train": len(set(groups[train_idx])),
        "clusters_test": len(set(groups[test_idx])),
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "mean_max_char_similarity_test_to_train": mean_max_sim,
        "test_ratio_max_sim_ge_0_90": leakage_090,
        "test_ratio_max_sim_ge_0_95": leakage_095,
    })

    test_probs = None
    if hasattr(clf, "predict_proba"):
        prob_matrix = clf.predict_proba(X_test)
        class_to_idx = {c: i for i, c in enumerate(clf.classes_)}
        test_probs = [
            float(prob_matrix[i, class_to_idx[y_pred[i]]])
            for i in range(len(y_pred))
        ]
    else:
        test_probs = [None] * len(y_pred)

    for local_i, global_i in enumerate(test_idx):
        oof_rows.append({
            "sample_id": int(df.iloc[global_i]["sample_id"]),
            "text": df.iloc[global_i][TEXT_COL],
            "true_label": y_test[local_i],
            "pred_label": y_pred[local_i],
            "pred_confidence": test_probs[local_i],
            "fold": fold_idx,
            "cluster_id": int(df.iloc[global_i]["cluster_id"]),
            "source_split": df.iloc[global_i]["source_split"],
            "max_char_similarity_to_train": float(max_sim_per_test[local_i]),
        })

    print(f"\n===== Fold {fold_idx} =====")
    print(f"Train size: {len(train_idx)} | Test size: {len(test_idx)}")
    print(f"Train clusters: {len(set(groups[train_idx]))} | Test clusters: {len(set(groups[test_idx]))}")
    print(f"Macro F1: {f1_macro:.4f} | Weighted F1: {f1_weighted:.4f} | Accuracy: {acc:.4f}")
    print(f"Mean max char-sim test->train: {mean_max_sim:.4f}")
    print(f"Ratio max sim >= 0.90: {leakage_090:.4f}")
    print(f"Ratio max sim >= 0.95: {leakage_095:.4f}")
    print(classification_report(y_test, y_pred, zero_division=0))


===== Fold 1 =====
Train size: 5201 | Test size: 1405
Train clusters: 2772 | Test clusters: 694
Macro F1: 0.9877 | Weighted F1: 0.9872 | Accuracy: 0.9872
Mean max char-sim test->train: 0.8196
Ratio max sim >= 0.90: 0.0000
Ratio max sim >= 0.95: 0.0000
              precision    recall  f1-score   support

           1       0.99      0.98      0.98       200
           2       0.98      0.98      0.98       391
           3       0.98      0.99      0.99       541
           4       1.00      0.99      1.00       273

    accuracy                           0.99      1405
   macro avg       0.99      0.99      0.99      1405
weighted avg       0.99      0.99      0.99      1405


===== Fold 2 =====
Train size: 5378 | Test size: 1228
Train clusters: 2773 | Test clusters: 693
Macro F1: 0.9800 | Weighted F1: 0.9788 | Accuracy: 0.9788
Mean max char-sim test->train: 0.8218
Ratio max sim >= 0.90: 0.0000
Ratio max sim >= 0.95: 0.0000
              precision    recall  f1-score   support

    

#Cell 5: Save results and show summary

In [ ]:
fold_metrics_df = pd.DataFrame(fold_metrics)
oof_df = pd.DataFrame(oof_rows).sort_values("sample_id").reset_index(drop=True)

overall_acc = accuracy_score(oof_df["true_label"], oof_df["pred_label"])
overall_precision_macro, overall_recall_macro, overall_f1_macro, _ = precision_recall_fscore_support(
    oof_df["true_label"], oof_df["pred_label"], average="macro", zero_division=0
)
overall_precision_weighted, overall_recall_weighted, overall_f1_weighted, _ = precision_recall_fscore_support(
    oof_df["true_label"], oof_df["pred_label"], average="weighted", zero_division=0
)

overall_metrics = pd.DataFrame([{
    "n_samples": len(df),
    "n_clusters": df["cluster_id"].nunique(),
    "similarity_threshold": SIM_THRESHOLD,
    "n_splits": N_SPLITS,
    "accuracy": overall_acc,
    "precision_macro": overall_precision_macro,
    "recall_macro": overall_recall_macro,
    "f1_macro": overall_f1_macro,
    "precision_weighted": overall_precision_weighted,
    "recall_weighted": overall_recall_weighted,
    "f1_weighted": overall_f1_weighted,
}])

fold_metrics_df.to_csv(OUTPUT_DIR / "fold_metrics.csv", index=False)
overall_metrics.to_csv(OUTPUT_DIR / "overall_metrics.csv", index=False)
oof_df.to_csv(OUTPUT_DIR / "oof_predictions.csv", index=False)

summary = {
    "n_samples": int(len(df)),
    "n_clusters": int(df["cluster_id"].nunique()),
    "cluster_size_top_20": {str(k): int(v) for k, v in cluster_sizes.head(20).to_dict().items()},
    "label_distribution": {str(k): int(v) for k, v in df[LABEL_COL].value_counts().to_dict().items()},
    "source_split_distribution": {str(k): int(v) for k, v in df["source_split"].value_counts().to_dict().items()},
    "similarity_threshold": SIM_THRESHOLD,
    "n_splits": N_SPLITS,
    "overall_metrics": overall_metrics.iloc[0].to_dict(),
}

with open(OUTPUT_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("===== Fold Metrics =====")
display(fold_metrics_df)

print("\n===== Overall Metrics =====")
display(overall_metrics)

print("\n===== Overall Classification Report =====")
print(classification_report(oof_df["true_label"], oof_df["pred_label"], zero_division=0))

print("\n===== Confusion Matrix =====")
labels_sorted = sorted(df[LABEL_COL].unique())
cm = confusion_matrix(oof_df["true_label"], oof_df["pred_label"], labels=labels_sorted)
cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in labels_sorted], columns=[f"pred_{x}" for x in labels_sorted])
display(cm_df)

print(f"\nSaved outputs to: {OUTPUT_DIR.resolve()}")

===== Fold Metrics =====


,fold,n_train,n_test,clusters_train,clusters_test,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,mean_max_char_similarity_test_to_train,test_ratio_max_sim_ge_0_90,test_ratio_max_sim_ge_0_95
0,1,5201,1405,2772,694,0.987189,0.988874,0.986560,0.987705,0.987222,0.987189,0.987196,0.819617,0.00000,0.00000
1,2,5378,1228,2773,693,0.978827,0.979519,0.980557,0.980010,0.978849,0.978827,0.978807,0.821794,0.00000,0.00000
2,3,5221,1385,2772,694,0.986282,0.986808,0.986355,0.986573,0.986303,0.986282,0.986284,0.824068,0.00000,0.00000
3,4,5387,1219,2771,695,0.985234,0.984140,0.985668,0.984887,0.985274,0.985234,0.985235,0.822814,0.00082,0.00082
4,5,5237,1369,2776,690,0.984660,0.982693,0.984574,0.983612,0.984682,0.984660,0.984657,0.824879,0.00000,0.00000



===== Overall Metrics =====


,n_samples,n_clusters,similarity_threshold,n_splits,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,6606,3466,0.88,5,0.984559,0.984525,0.98515,0.984836,0.984561,0.984559,0.984559



===== Overall Classification Report =====
              precision    recall  f1-score   support

           1       0.98      0.98      0.98       998
           2       0.98      0.98      0.98      1843
           3       0.99      0.98      0.98      2415
           4       0.99      0.99      0.99      1350

    accuracy                           0.98      6606
   macro avg       0.98      0.99      0.98      6606
weighted avg       0.98      0.98      0.98      6606


===== Confusion Matrix =====


,pred_1,pred_2,pred_3,pred_4
true_1,981,11,5,1
true_2,11,1805,25,2
true_3,7,27,2375,6
true_4,2,1,4,1343



Saved outputs to: /content/tfidf_cluster_aware_outputs


#CELL 6

In [ ]:
# =========================
# Save TF-IDF evaluation metrics
# =========================

tfidf_dir = OUTPUT_DIR / "tfidf_metrics"
tfidf_dir.mkdir(exist_ok=True, parents=True)

# 1. save fold metrics
fold_metrics_df.to_csv(
    tfidf_dir / "tfidf_fold_metrics.csv",
    index=False
)

# 2. save overall metrics
overall_metrics.to_csv(
    tfidf_dir / "tfidf_overall_metrics.csv",
    index=False
)

# 3. save out-of-fold predictions
oof_df.to_csv(
    tfidf_dir / "tfidf_oof_predictions.csv",
    index=False
)

print("\nTF-IDF metrics exported to:")
print(tfidf_dir.resolve())


TF-IDF metrics exported to:
/content/tfidf_cluster_aware_outputs/tfidf_metrics
